<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

# Citysearch Agent deployed on Agentcore runtime
In this notebook, we set recreate simple chat bot we built with Strands in 03-agentic-metrics, to be deployable with AgentCore Runtime.  

## 1) Set up dependances

In [1]:
# Install Strands Agents
!pip install strands-agents strands-agents-tools pandas bedrock_agentcore_starter_toolkit

Defaulting to user installation because normal site-packages is not writeable


  Using cached boto3-1.42.88-py3-none-any.whl.metadata (6.7 kB)


  Using cached botocore-1.42.88-py3-none-any.whl.metadata (5.9 kB)


  Using cached s3transfer-0.16.0-py3-none-any.whl.metadata (1.7 kB)


  Using cached jsonschema-4.24.1-py3-none-any.whl.metadata (7.5 kB)


...

    Found existing installation: jsonschema 4.23.0
    Uninstalling jsonschema-4.23.0:
      Successfully uninstalled jsonschema-4.23.0


  Attempting uninstall: boto3


    Found existing installation: boto3 1.40.61
    Uninstalling boto3-1.40.61:
      Successfully uninstalled boto3-1.40.61


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.25.1 requires botocore<1.40.62,>=1.40.46, but you have botocore 1.42.88 which is incompatible.
litellm 1.83.4 requires jsonschema==4.23.0, but you have jsonschema 4.24.1 which is incompatible.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


We'll also import a list of city data, to use as our gold standard set.
This is from: https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population

Now we'll make a quick checking function.  Our agent will be trying to find population and area, so we'll compare how well it did against this list.

## 1) Lets rebuilt the a simple chat bot from 03-agentic-metrics to be deployed with AgentCore

In [2]:
%%writefile ./citysearch.py
from botocore.config import Config
import requests
from ddgs import DDGS
from strands import Agent, tool
from strands.models import BedrockModel
import pandas as pd
import re

#A custom config for Bedrock to only allow short connections - for our demo we expect all calls to be fast.
#here we turn off retries, and we time out after 20 seconds.
quick_config = Config(
    connect_timeout=5,
    read_timeout=20,
    retries={"max_attempts": 0}
)

from bedrock_agentcore.runtime import (
    BedrockAgentCoreApp,
)  #### AGENTCORE RUNTIME - LINE 1 ####
from strands import Agent
from strands.models import BedrockModel

from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

@tool
def web_search(topic: str) -> str:
    """Search Duck Duck go Service for a given topic.
    Return a string listing the top 5 results including the url, title, and description of each result.
    """
    try:
        results = DDGS(timeout=5).text(topic, region=region, max_results=5)
        return results if results else "No results found."
    except Exception as e:
        return f"Search error: {str(e)}"

#Create the chatbot.  We'll use Nova Micro to optimize for latency, cost, and capacity
chatbot_model_name = "us.amazon.nova-micro-v1:0"
#add custom timeout for the model, to keep the tool from hanging or retrying too much.
chatbot_model = BedrockModel(
    model_id=chatbot_model_name,
    boto_client_config=quick_config    
)
chatbot = Agent(tools=[web_search], model=chatbot_model)
#Call the chat bot with a simple request.
prompt = """How many people live in New York, and what's the area of the city in square miles?
After you respond, also include your answer in 'pop' and 'area' XML tags, for programatic processing.
The values in the XML tags should only be numbers, no words or commas."""
chatbot_response = chatbot(prompt)

# Initialize the AgentCore Runtime App
app = BedrockAgentCoreApp()  #### AGENTCORE RUNTIME - LINE 2 ####

@app.entrypoint  #### AGENTCORE RUNTIME - LINE 3 ####
def invoke(payload):
    """AgentCore Runtime entrypoint function"""
    user_input = payload.get("prompt", "")

    # Invoke the agent
    response = chatbot(user_input)
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    app.run()  #### AGENTCORE RUNTIME - LINE 4 ####


Overwriting ./citysearch.py


# 2) Lets configure the agent with AgentCore Runtime

In [3]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "citysearch"
response = agentcore_runtime.configure(
    entrypoint="citysearch.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="05-02-03-requirements.txt",
    region=region,
    agent_name=agent_name
)
response

Entrypoint parsed: file=/local/home/lukewma/evals-workshop/05-framework-specific-evaluations/05-02-AgentCore/citysearch.py, bedrock_agentcore_name=citysearch


Memory disabled - agent will be stateless


Configuring BedrockAgentCore agent: citysearch


Memory disabled


Network mode: PUBLIC


...

📄 Using existing Dockerfile: 
/local/home/lukewma/evals-workshop/05-framework-specific-evaluations/05-02-AgentCore/Dockerfile

Generated .dockerignore: /local/home/lukewma/evals-workshop/05-framework-specific-evaluations/05-02-AgentCore/.dockerignore


Keeping 'citysearch' as default agent


Bedrock AgentCore configured: /local/home/lukewma/evals-workshop/05-framework-specific-evaluations/05-02-AgentCore/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/local/home/lukewma/evals-workshop/05-framework-specific-evaluations/05-02-AgentCore/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/local/home/lukewma/evals-workshop/05-framework-specific-evaluations/05-02-AgentCore/Dockerfile'), dockerignore_path=PosixPath('/local/home/lukewma/evals-workshop/05-framework-specific-evaluations/05-02-AgentCore/.dockerignore'), runtime='Docker', runtime_type=None, region='us-west-2', account_id='762978522101', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

# 3) Now that the citysearch agent is configured with AgentCore, lets launch it. 
This step 
1. Publishes the Docker file as a container into ECR, 
2. Creates a new generates an version for the agent. If you run this multiple times you will see multiple versions of the Agent. If you do not intend to update existing agent, set the auto_update_on_conflict=False
3. Deploys the container on AgentCore Runtime
This creates / updates the AgentCore runtime for the agent "citysearch" to have the latest version of the agent.

In [4]:
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...


   • Deploy Python code directly to runtime


   • No Docker required (DEFAULT behavior)


   • Production-ready deployment


💡 Deployment options:


...

🔍 Agent logs available at:


   /aws/bedrock-agentcore/runtimes/citysearch-nv57Ez86iG-DEFAULT --log-stream-name-prefix "2026/04/12/\[runtime-logs]"


   /aws/bedrock-agentcore/runtimes/citysearch-nv57Ez86iG-DEFAULT --log-stream-names "otel-rt-logs"


💡 Tail logs with: aws logs tail /aws/bedrock-agentcore/runtimes/citysearch-nv57Ez86iG-DEFAULT --log-stream-name-prefix "2026/04/12/\[runtime-logs]" --follow


💡 Or view recent logs: aws logs tail /aws/bedrock-agentcore/runtimes/citysearch-nv57Ez86iG-DEFAULT --log-stream-name-prefix "2026/04/12/\[runtime-logs]" --since 1h


# 4) Check the status of the deployed agent to ensure agent is ready for inference at the endpoint

In [5]:
## Check agent status
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

Retrieved Bedrock AgentCore status for: citysearch


'READY'

# 5) Test the agent to ensure it is ready for evaluation

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
## Test successful deployment of the agent
import boto3
import json
import uuid
import textwrap

# Create a session ID for demonstrating session continuity with AgentCore
session_id = uuid.uuid4()

client = boto3.client('bedrock-agentcore', region_name='us-east-1')
payload =  json.dumps({"prompt": "How many people live in New York, and what's the area of the city in square miles?"})

response = client.invoke_agent_runtime(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=f"eval-session-{uuid.uuid4()}",  # Must be 33+ chars
    payload=payload,
    qualifier="DEFAULT" # Optional
)
response_body = response['response'].read()
response_data = json.loads(response_body)
formatted = json.dumps(response_data, indent=2)
wrapped = textwrap.fill(formatted, width=80)
print(wrapped)



ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the InvokeAgentRuntime operation: No endpoint or agent found with qualifier 'DEFAULT' for agent 'arn:aws:bedrock-agentcore:us-west-2:762978522101:runtime/citysearch-nv57Ez86iG'

In [ ]:
citysearch_agent_arn=launch_result.agent_arn
%store citysearch_agent_arn


In [ ]:
## Lets look at the output of the agent
response

## Here is what we have accomplished so far:

1. We upgraded an existing agent to be deployable on Agentcore runtime
2. Tested a simple invocation of the citysearch agent running on Agentcore
3. Ensured the tool is appropriately configured through Agent Observability logs. Refer to README for screenshots which you can observe on your AWS console
4. Stored the agent arn to be able to retrieve during the agent evaluations in the next notebook 

## Now lets move on to the next portion which is agent evaluations --> 05-02-02-Agent-and-tool-evals-with-cwlogs.ipynb

## Here is what we have accomplished so far:

1. We upgraded an existing agent to be deployable on Agentcore runtime
2. Tested a simple invocation of the citysearch agent running on Agentcore
3. Ensured the tool is appropriately configured through Agent Observability logs. Refer to README for screenshots which you can observe on your AWS console
4. Stored the agent arn to be able to retrieve during the agent evaluations in the next notebook 

## Now lets move on to the next portion which is agent evaluations --> 05-02-02-Agent-and-tool-evals-with-cwlogs.ipynb